In [1]:
# ═══════════════════════════════════════════════════════════════
#  TRPG 调查员助手 —— 主流程 Notebook (Multi-Agent 架构)
# ═══════════════════════════════════════════════════════════════

import sys
import json
from datetime import datetime
from IPython.display import HTML, display

# 将 src/ 加入路径以导入依赖模块
sys.path.insert(0, "../src")

from game_loop import init_game, run_turn
from llm import set_llm_log_file
from prompts import set_prompt_log_file
from library import WeaponLibrary, EnemyLibrary, ContentInjector
from trpg_display import (
    display_narrative, display_scene, display_system, display_debug,
    display_input_area, render_scene_to_html, display_split_result,
)

# ── COC 7th 车卡系统 ──
from investigator import Investigator, load_investigator
from investigator.rules import roll_stats, calc_derived, create_skill_list

In [2]:
# ═══════════════════════════════════════════════════════════════
#  Prompt 日志配置
# ═══════════════════════════════════════════════════════════════

PROMPT_LOG_FILE = f"../logs/prompt_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
set_prompt_log_file(PROMPT_LOG_FILE)
set_llm_log_file(PROMPT_LOG_FILE)

In [3]:
# ============================================================
#  武器/敌人库初始化（新增 — parser 系统升级）
# ============================================================

weapon_lib = WeaponLibrary()
weapon_lib.load_core()
enemy_lib = EnemyLibrary()
enemy_lib.load_core()
injector = ContentInjector(weapon_lib, enemy_lib)
display_system(
    f"武器库：{len(weapon_lib)} 件 | 敌人库：{len(enemy_lib)} 个 | "
    f"注入器：{"就绪" if injector else "未初始化"}",
    "info"
)

In [4]:
def run_game(character_path: str = None):
    """
    启动 TRPG 游戏主循环 (Multi-Agent 架构)。

    参数:
        character_path: 调查员 JSON 文件路径（可选）。
    """
    import json as _json
    import os as _os

    # ── 使用新多 Agent 入口初始化 ──
    game = init_game(
        l2_path="../data/modules/常暗之厢/l2_keeper.json",
        l1_path="../data/modules/常暗之厢/l1_player.json",
        l3_path="../data/modules/常暗之厢/l3_designer.json",
        escalation_config_path="../data/modules/常暗之厢/escalation_config.json",
        start_node="6号车厢",
    )

    keeper = game["keeper"]
    world = keeper.world
    print(f"场景数：{len(world.graph.nodes)}, 事件数：{len(world.graph.events)}")

    # ── 加载调查员（COC 7th 车卡系统）──
    if character_path is None:
        character_path = "../investigator/test_character.json"

    if _os.path.exists(character_path):
        investigator = load_investigator(character_path)
        display_system(
            f"已加载调查员：{investigator.name} | "
            f"职业：{investigator.occupation.name if investigator.occupation else '无'} | "
            f"HP={investigator.derived.HP} SAN={investigator.derived.SAN}",
            "info"
        )
    else:
        display_system(
            f"未找到角色卡文件 {character_path}，正在掷骰生成默认调查员...",
            "warn"
        )
        investigator = Investigator(name="调查员A", age=25, gender="男")
        investigator.stats = roll_stats()
        investigator.skills = create_skill_list()
        investigator.derived = calc_derived(investigator.stats, investigator.age)
        display_system(
            f"已生成调查员：{investigator.name} | "
            f"HP={investigator.derived.HP} SAN={investigator.derived.SAN}",
            "info"
        )

    world.set_player(investigator)

    # ── 确保存档目录存在 ──
    _os.makedirs("../data/saves", exist_ok=True)

    # ── 开场 ──
    display_system("游戏开始。输入 /help 查看可用命令。", "info")
    display(HTML(render_scene_to_html(world)))

    # 开场叙事
    initial = run_turn(game, "（游戏开始）")
    display_split_result(initial["brief"], initial["narrative"])

    # ── 主循环 ──
    while True:
        cmd = input("\n> ").strip()
        if not cmd:
            continue

        if cmd in ("exit", "quit"):
            display_system("游戏结束。", "info")
            break
        elif cmd.startswith("/scene"):
            display(HTML(render_scene_to_html(world)))
            continue
        elif cmd.startswith("/info"):
            display_system(json.dumps(world.get_scene_info(), ensure_ascii=False, indent=2), "debug")
            continue
        elif cmd.startswith("/events"):
            active = world.get_active_event_effects()
            if active:
                for name, impact in active:
                    display_system(f"◆ {name}\n  {impact}", "info")
            else:
                display_system("（无已触发事件）", "info")
            continue
        elif cmd.startswith("/flags"):
            display_system(str(dict(world.flags)), "debug")
            continue
        elif cmd.startswith("/char"):
            if world.player:
                display_system(str(world.player), "debug")
            else:
                display_system("（未设置调查员）", "warn")
            continue
        elif cmd.startswith("/save"):
            slot = cmd.split(maxsplit=1)[1] if len(cmd.split()) > 1 else "quick"
            path = f"../data/saves/{slot}.json"
            world.save_state(path)
            display_system(f"存档已保存至 {path}", "info")
            continue
        elif cmd.startswith("/load"):
            slot = cmd.split(maxsplit=1)[1] if len(cmd.split()) > 1 else "quick"
            path = f"../data/saves/{slot}.json"
            if _os.path.exists(path):
                from scenario_core import ScenarioWorld
                new_world = ScenarioWorld.load_state(path)
                keeper.world = new_world
                world = new_world
                display_system(f"已从 {path} 读档", "info")
                display(HTML(render_scene_to_html(world)))
            else:
                display_system(f"存档 {path} 不存在", "warn")
            continue
        elif cmd.startswith("/help"):
            display_system(
                "/scene 场景 | /info 状态 | /events 事件 | /flags 标记\n"
                "/char 角色 | /do <动作> | /trigger <E1> | /spawn enemy/weapon <名称>\n"
                "/save <槽位> | /load <槽位> | /charsave | /charload | exit",
                "info"
            )
            continue

        # 正常回合
        result = run_turn(game, cmd)
        display_split_result(result["brief"], result["narrative"])

In [5]:
run_game()

场景数：7, 事件数：26


AttributeError: 'Entity' object has no attribute 'summary'